In [ ]:
from transformers import LlamaModel

/home/user-name-goes-here/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'LlamaForCalusalLM' from 'transformers' (/home/user-name-goes-here/.local/lib/python3.11/site-packages/transformers/__init__.py)

In [1]:
import torch
import torch.nn as nn

# --- 1. Настройки и инициализация случайных данных ---
# Используем небольшие, но нетривиальные размеры для демонстрации
BATCH_SIZE = 4
SEQ_LEN = 512
HIDDEN_DIM = 128
VOCAB_SIZE = 1000
NUM_MASKED_TOKENS = 15  # Количество токенов, для которых считаем loss

print(f"Параметры симуляции:")
print(f"Batch Size: {BATCH_SIZE}, Sequence Length: {SEQ_LEN}")
print(f"Hidden Dim: {HIDDEN_DIM}, Vocab Size: {VOCAB_SIZE}")
print(f"Токенов для loss: {NUM_MASKED_TOKENS}")
print("-" * 30)

# Для воспроизводимости результатов
torch.manual_seed(42)

# Создаем случайные входные данные
hidden_states = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM)
labels = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
# Создаем "голову" модели - обычный линейный слой
lm_head = nn.Linear(HIDDEN_DIM, VOCAB_SIZE)
# Функция потерь
loss_fct = nn.CrossEntropyLoss()

# --- 2. Создание случайной маски ---
# Стандартная практика в LLM - сдвигать метки и логиты, чтобы предсказывать следующий токен
# Маска должна иметь размерность [BATCH_SIZE, SEQ_LEN - 1]
shift_mask = torch.zeros(BATCH_SIZE, SEQ_LEN - 1, dtype=torch.bool)
# Выбираем NUM_MASKED_TOKENS случайных позиций в батче, где маска будет True
for _ in range(NUM_MASKED_TOKENS):
    b = torch.randint(0, BATCH_SIZE, (1,)).item()
    s = torch.randint(0, SEQ_LEN - 1, (1,)).item()
    # Убедимся, что не выбрали одну и ту же позицию дважды
    while shift_mask[b, s]:
        b = torch.randint(0, BATCH_SIZE, (1,)).item()
        s = torch.randint(0, SEQ_LEN - 1, (1,)).item()
    shift_mask[b, s] = True

# --- 3. Неэффективный подход (фильтрация ПОСЛЕ lm_head) ---
print("\n--- [Подход 1] Неэффективный (фильтрация логитов) ---")

# Полное вычисление логитов
lm_logits_old = lm_head(hidden_states)

# Сдвиг логитов и меток
shift_logits_old = lm_logits_old[:, :-1, :]
shift_labels_old = labels[:, 1:]

# "Выравнивание" тензоров. .contiguous() здесь важен для предсказуемого порядка!
flat_logits_old = shift_logits_old.contiguous().view(-1, VOCAB_SIZE)
flat_labels_old = shift_labels_old.contiguous().view(-1)
flat_mask = shift_mask.view(-1)

# Применение маски к уже вычисленным логитам
masked_logits_old = flat_logits_old[flat_mask]
masked_labels_old = flat_labels_old[flat_mask]

# Расчет Loss
loss_old = loss_fct(masked_logits_old, masked_labels_old)

print(f"Форма masked_logits: {masked_logits_old.shape}")
print(f"Loss: {loss_old.item():.6f}")

# --- 4. Оптимизированный подход (фильтрация ДО lm_head) ---
print("\n--- [Подход 2] Оптимизированный (фильтрация hidden_states) ---")

# Сдвиг скрытых состояний и меток
shift_hidden_states_new = hidden_states[:, :-1, :]
shift_labels_new = labels[:, 1:]

# "Выравнивание" тензоров. .contiguous() снова критически важен!
flat_hidden_states_new = shift_hidden_states_new.contiguous().view(-1, HIDDEN_DIM)
flat_labels_new = shift_labels_new.contiguous().view(-1)
# flat_mask уже создан и он тот же самый

# Применение маски к скрытым состояниям
masked_hidden_states_new = flat_hidden_states_new[flat_mask]
masked_labels_new = flat_labels_new[flat_mask]

# Вычисление логитов ТОЛЬКО для нужных 15 токенов
masked_logits_new = lm_head(masked_hidden_states_new)

# Расчет Loss
loss_new = loss_fct(masked_logits_new, masked_labels_new)

print(f"Форма masked_logits: {masked_logits_new.shape}")
print(f"Loss: {loss_new.item():.6f}")


# --- 5. Проверка гипотезы и вердикт ---
print("\n--- [Вердикт] Сравнение результатов ---")

# Проверяем, что выбранные метки полностью совпадают
labels_are_equal = torch.equal(masked_labels_old, masked_labels_new)
print(f"1. Выбранные метки (labels) идентичны: {labels_are_equal}")
if not labels_are_equal:
    print("   -> ОШИБКА: Проблема в логике создания маски или нарезки меток.")

# Проверяем, что логиты практически идентичны (используем allclose для float)
logits_are_close = torch.allclose(masked_logits_old, masked_logits_new, atol=1e-6)
print(f"2. Вычисленные логиты (logits) идентичны: {logits_are_close}")
if not logits_are_close:
    print(
        "   -> ОШИБКА: Самая вероятная причина - пропущенный .contiguous() перед .view(),"
    )
    print("      что привело к разному порядку элементов в 'выровненных' тензорах.")

# Проверяем, что итоговый loss практически идентичен
loss_is_close = torch.allclose(loss_old, loss_new, atol=1e-6)
print(f"3. Итоговый loss идентичен: {loss_is_close}")

print("-" * 30)
if labels_are_equal and logits_are_close and loss_is_close:
    print(
        "✅ Гипотеза подтверждена! Оба подхода дают математически эквивалентный результат."
    )
    print("   Оптимизация безопасна и корректна.")
else:
    print(
        "❌ Гипотеза не подтверждена. В коде есть ошибка, которая приводит к расхождению."
    )

Параметры симуляции:
Batch Size: 4, Sequence Length: 512
Hidden Dim: 128, Vocab Size: 1000
Токенов для loss: 15
------------------------------

--- [Подход 1] Неэффективный (фильтрация логитов) ---
Форма masked_logits: torch.Size([15, 1000])
Loss: 7.084174

--- [Подход 2] Оптимизированный (фильтрация hidden_states) ---
Форма masked_logits: torch.Size([15, 1000])
Loss: 7.084174

--- [Вердикт] Сравнение результатов ---
1. Выбранные метки (labels) идентичны: True
2. Вычисленные логиты (logits) идентичны: True
3. Итоговый loss идентичен: True
------------------------------
✅ Гипотеза подтверждена! Оба подхода дают математически эквивалентный результат.
   Оптимизация безопасна и корректна.
